# Keras/TensorFlow — Chapter 3: Using Autograd to Solve a Regression Problem


## 1. Tensor và Variable

- `tf.constant([1,2,3])` — tensor bất biến.
- `tf.Variable([1,2,3])` — biến có thể cập nhật giá trị qua các bước huấn luyện.

## 2. GradientTape

- Khai báo **tường minh** một block `with tf.GradientTape() as tape:` bao quanh đoạn forward cần lấy đạo hàm.
- Sau block đó, gọi `tape.gradient(y, x)` để lấy đạo hàm, trả gradient trực tiếp qua giá trị hàm trả về.
- Ví dụ `y = x*x` tại `x = 3.6` → `dy/dx = 7.2` (`tf.Tensor(7.2, shape=(), dtype=float32)`).

## 3. Hồi quy đa thức bằng GradientTape

- Mỗi vòng lặp phải mở lại `with tf.GradientTape() as tape:`.
- `tf.reduce_sum(tf.square(y - y_pred))` — dùng **tổng** bình phương sai số.
- Cập nhật trọng số qua `optimizer.apply_gradients([(grad, w)])`.

## 4. Giải hệ phương trình bằng GradientTape

Hệ phương trình trong sách Keras: `A+B=8, C-D=6, A+C=13, B+D=8`
- `tape.gradient(sqerr, [A,B,C,D])` trả về **list gradient cho nhiều biến cùng lúc**. Lý do bắt buộc gọi 1 lần duy nhất: mặc định, gradient của `sqerr` chỉ truy xuất được **một lần** từ tape — muốn gọi `tape.gradient()` nhiều lần cho cùng một phép tính phải khai báo `persistent=True`.

## 5. Vận dụng


Tạo tensor bất biến bằng tf.constant

In [ ]:
import tensorflow as tf

x = tf.constant([1, 2, 3])
print(x)
print(x.shape)
print(x.dtype)


Tạo biến có thể cập nhật bằng tf.Variable

In [ ]:
import tensorflow as tf

x = tf.Variable([1, 2, 3])
print(x)
print(x.shape)
print(x.dtype)


Tính đạo hàm bằng GradientTape

In [ ]:
import tensorflow as tf

x = tf.Variable(3.6)

with tf.GradientTape() as tape:
    y = x*x

dy = tape.gradient(y, x)
print(dy)


Tạo đa thức bậc 2 bằng NumPy poly1d

In [ ]:
import numpy as np

polynomial = np.poly1d([1, 2, 3])
print(polynomial)


Hồi quy đa thức bằng GradientTape + optimizer Nadam

In [ ]:
import numpy as np
import tensorflow as tf

N = 20   # number of samples

# Generate random samples between -10 to +10
polynomial = np.poly1d([1, 2, 3])
X = np.random.uniform(-10, 10, size=(N,1))
Y = polynomial(X)

# Prepare input as an array of shape (N,3)
XX = np.hstack([X*X, X, np.ones_like(X)])

# Prepare TensorFlow objects
w = tf.Variable(tf.random.normal((3,1)))  # the 3 coefficients
x = tf.constant(XX, dtype=tf.float32)     # input sample
y = tf.constant(Y, dtype=tf.float32)      # output sample
optimizer = tf.keras.optimizers.Nadam(learning_rate=0.01)
print(w)

# Run optimizer
for _ in range(1000):
    with tf.GradientTape() as tape:
        y_pred = x @ w
        mse = tf.reduce_sum(tf.square(y - y_pred))
    grad = tape.gradient(mse, w)
    optimizer.apply_gradients([(grad, w)])

print(w)


Giải hệ phương trình 4 ẩn bằng GradientTape

In [ ]:
import tensorflow as tf
import random

A = tf.Variable(random.random())
B = tf.Variable(random.random())
C = tf.Variable(random.random())
D = tf.Variable(random.random())

# Gradient descent loop
EPOCHS = 1000
optimizer = tf.keras.optimizers.Nadam(learning_rate=0.1)
for _ in range(EPOCHS):
    with tf.GradientTape() as tape:
        y1 = A + B - 8
        y2 = C - D - 6
        y3 = A + C - 13
        y4 = B + D - 8
        sqerr = y1*y1 + y2*y2 + y3*y3 + y4*y4
    gradA, gradB, gradC, gradD = tape.gradient(sqerr, [A, B, C, D])
    optimizer.apply_gradients([(gradA, A), (gradB, B), (gradC, C), (gradD, D)])

print(A)
print(B)
print(C)
print(D)
